# Setup
## Necessary library imports

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from tabulate import tabulate
import numpy as np

# Extract 
## Getting the players, player_stats, teams and plays data from those CSV files

In [ ]:
#Get players
file_path = "base_data/players.csv"

players_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get players stats
file_path = "playerStats_data/playerStats_2024_ENG.1.csv"

player_stats_2024_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get teams
file_path = "base_data/teams.csv"

teams_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get plays
file_path = "plays_data/plays_2024_ENG.1.csv"

plays_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

# player_stats_2024_df = player_stats_2024_df.sort_values(by="shotsFaced_value", ascending=False)
# print(tabulate(player_stats_2024_df.head(99), headers="keys", tablefmt='psql'))

100%|██████████| 11.3M/11.3M [00:03<00:00, 3.69MB/s]
/workspaces/Helix-Football-Data-App/myenv/lib/python3.12/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (0: nickName) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


100%|██████████| 64.0k/64.0k [00:00<00:00, 109kB/s]


100%|██████████| 508k/508k [00:01<00:00, 292kB/s]


100%|██████████| 11.3M/11.3M [00:03<00:00, 3.68MB/s]


# Transform
## Creating a unified dataframe with combined stats, calculated metrics, and percentiles

In [3]:
# Drop columns I won't be using in each df

players_df = players_df[[
    "athleteId",
    "fullName",
    "slug",
    "weight",
    "displayHeight",
    "height",
    "age",
    "dateOfBirth",
    "citizenship",
    "positionAbbreviation"
]]

player_stats_2024_df = player_stats_2024_df[[
    "teamId",
    "athleteId",
    "appearances_value",
    "foulsCommitted_value",
    "foulsSuffered_value",
    "yellowCards_value",
    "redCards_value",
    "ownGoals_value",
    "goalAssists_value",
    "offsides_value",
    "shotsOnTarget_value",
    "totalShots_value",
    "totalGoals_value",
    "shotsFaced_value",
    "saves_value",
    "goalsConceded_value"
]]

# Getting English first division team IDs so that I can isolate them in the teams dataframe
prem_teams_ids = player_stats_2024_df["teamId"].unique().tolist()

# Creating a dataframe tha's only English first divisiion teams
prem_teams_df = teams_df[teams_df["teamId"].isin(prem_teams_ids)]

# Merge players.csv and player_stats.csv on athleteId
first_merged_df = players_df.merge(player_stats_2024_df, on="athleteId")

# Merge unified players dataframes with teams dataframe so that team name is added as a column
second_merged_df = first_merged_df.merge(teams_df[["teamId", "displayName"]], on="teamId", how="left")

# Rename displayName to something clearer
second_merged_df = second_merged_df.rename(columns={"displayName":"teamName"})

# print(tabulate(second_merged_df.head(300), headers="keys", tablefmt='psql'))
# print(tabulate(prem_teams_df.head(300), headers="keys", tablefmt='psql'))
# print(second_merged_df.columns)

In [4]:
# Adding calculated stats columns using numpy

unified_df = second_merged_df.copy()
print(unified_df.columns)

# shot accuracy
unified_df["shot_accuracy"] = np.where (
    unified_df["totalShots_value"] > 0,
    unified_df["shotsOnTarget_value"] / unified_df["totalShots_value"],
    np.nan
)

# conversion rate
unified_df["conversion_rate"] = np.where (
    unified_df["totalShots_value"] > 0,
    unified_df["totalGoals_value"] / unified_df["totalShots_value"],
    np.nan
)

# on target conversion rate
unified_df["on_target_conversion_rate"] = np.where (
    unified_df["shotsOnTarget_value"] > 0,
    unified_df["totalGoals_value"] / unified_df["shotsOnTarget_value"],
    np.nan
)

# save percentage
unified_df["save_percentage"] = np.where (
    (unified_df["saves_value"] + unified_df["goalsConceded_value"] > 0) & (unified_df["positionAbbreviation"]=="G"),
    unified_df["saves_value"] / (unified_df["saves_value"]+ unified_df["goalsConceded_value"]),
    np.nan
)

# discipline score
unified_df["discipline_score"] = unified_df["yellowCards_value"] + (unified_df["redCards_value"] * 2)

# unified_df.head()
print(tabulate(unified_df.head(20), headers="keys", tablefmt='psql'))



Index(['athleteId', 'fullName', 'slug', 'weight', 'displayHeight', 'height',
       'age', 'dateOfBirth', 'citizenship', 'positionAbbreviation', 'teamId',
       'appearances_value', 'foulsCommitted_value', 'foulsSuffered_value',
       'yellowCards_value', 'redCards_value', 'ownGoals_value',
       'goalAssists_value', 'offsides_value', 'shotsOnTarget_value',
       'totalShots_value', 'totalGoals_value', 'shotsFaced_value',
       'saves_value', 'goalsConceded_value', 'teamName'],
      dtype='str')
+----+-------------+------------------+------------------+----------+-----------------+----------+-------+-------------------+---------------------+------------------------+----------+---------------------+------------------------+-----------------------+---------------------+------------------+------------------+---------------------+------------------+-----------------------+--------------------+--------------------+--------------------+---------------+-----------------------+----------

In [8]:
# Percentile calculation

# Percentile compared to players of the same position
def add_percentile_by_position(dataframe, column_name, new_column):
    for position in ["M","F","D", "G"]:
        position_match = (dataframe["positionAbbreviation"] == position)
        # choosing rows where the position is matched and grabbing column name
        # then ranking them based on percentile        
        dataframe.loc[position_match, new_column] = dataframe.loc[position_match, column_name].rank(pct=True) * 100
    return dataframe

# Percentile compared to every player
def add_percentile_global(dataframe, column_name, new_column):
    dataframe[new_column] = dataframe[column_name].rank(pct=True) * 100
    return dataframe

In [9]:
# Testing positional

test_df = add_percentile_by_position(unified_df, "totalGoals_value", "goals_percentile")
forwards = test_df[test_df["positionAbbreviation"] == "F"]
print(forwards[['fullName', 'totalGoals_value', 'goals_percentile']].sort_values('goals_percentile', ascending=False).head(10))
print()

# Testing global
test_df = add_percentile_global(test_df, "totalGoals_value", "goals_percentile")
print(test_df[["fullName", "positionAbbreviation", "totalGoals_value", "goals_percentile"]].sort_values("goals_percentile", ascending=False).head(10))


                 fullName  totalGoals_value  goals_percentile
141         Mohamed Salah                29        100.000000
315        Alexander Isak                23         99.502488
381        Erling Haaland                22         99.004975
435          Bryan Mbeumo                20         98.258706
53             Chris Wood                20         98.258706
264           Yoane Wissa                19         97.512438
197         Ollie Watkins                16         97.014925
402         Matheus Cunha                15         96.517413
337  Jean-Philippe Mateta                14         95.771144
379  Jørgen Strand Larsen                14         95.771144

                 fullName positionAbbreviation  totalGoals_value  \
141         Mohamed Salah                    F                29   
315        Alexander Isak                    F                23   
381        Erling Haaland                    F                22   
435          Bryan Mbeumo                    

In [12]:
# Creating position specific percentiles for outfield players (for my radar charts)

unified_df = add_percentile_by_position(unified_df, "totalGoals_value", "goals_pct_pos")
unified_df = add_percentile_by_position(unified_df, "goalAssists_value", "assists_pct_pos")
unified_df = add_percentile_by_position(unified_df, "shot_accuracy", "shot_accuracy_pct_pos")
unified_df = add_percentile_by_position(unified_df, "foulsSuffered_value", "fouls_suff_pct_pos")
unified_df = add_percentile_by_position(unified_df, "foulsCommitted_value", "fouls_comm_pct_pos")

# For forwards
unified_df = add_percentile_by_position(unified_df, 'conversion_rate', 'conversion_pct_pos')
unified_df = add_percentile_by_position(unified_df, 'on_target_conversion_rate', 'on_target_conv_pct_pos')

# Creating global percentiles for leaderboards
unified_df = add_percentile_global(unified_df, "totalGoals_value", "goals_pct_global")
unified_df = add_percentile_global(unified_df, "goalAssists_value", "assists_pct_global")
unified_df = add_percentile_global(unified_df, "shot_accuracy", "shot_accuracy_pct_global")
unified_df = add_percentile_global(unified_df, "foulsSuffered_value", "fouls_suff_pct_global")
unified_df = add_percentile_global(unified_df, "foulsCommitted_value", "fouls_comm_pct_global")

# Inverted metrics

# GK metrics
unified_df = add_percentile_by_position(unified_df, 'goalsConceded_value', 'goals_conc_pct_pos')
unified_df['goals_conc_pct_pos'] = 100 - unified_df['goals_conc_pct_pos']

unified_df = add_percentile_by_position(unified_df, 'save_percentage', 'save_pct_pos')
unified_df = add_percentile_by_position(unified_df, 'saves_value', 'saves_pct_pos')

# Discipline metrics (defenders)
unified_df = add_percentile_by_position(unified_df, 'yellowCards_value', 'yellow_cards_pct_pos')
unified_df['yellow_cards_pct_pos'] = 100 - unified_df['yellow_cards_pct_pos']

unified_df = add_percentile_by_position(unified_df, 'redCards_value', 'red_cards_pct_pos')
unified_df['red_cards_pct_pos'] = 100 - unified_df['red_cards_pct_pos']

# Offsides (forwards)
unified_df = add_percentile_by_position(unified_df, 'offsides_value', 'offsides_pct_pos')
unified_df['offsides_pct_pos'] = 100 - unified_df['offsides_pct_pos']

# Load
## Storing the unified dataframe as a CSV file to be accessed later

In [14]:
# Saving the unified df as a CSV

unified_df.to_csv('unified_players_ENG_1_2024.csv', index=False)